In [ ]:
import pandas as pd
from rdkit import Chem
from chembl_webresource_client.new_client import new_client

In [ ]:
activity = new_client.activity

target_chembl_id = 'CHEMBL230'

data = activity.filter(target_chembl_id=target_chembl_id).filter(standard_type='IC50')

df = pd.DataFrame.from_records(data)

print(f'Загружено записей: {len(df)}')

df.to_csv('cox2_raw.csv', index=False)
print("Сырые данные сохранены в 'cox2_raw.csv'")

Загружено записей: 7980
Сырые данные сохранены в 'cox2_raw.csv'


In [2]:
# --- 1. Загрузка датасета ---

df = pd.read_csv('cox2_raw.csv')
print("Размер исходного датасета:", df.shape)

Размер исходного датасета: (7980, 46)


In [ ]:
# --- 2. Оставляем только релевантные записи ---

columns_to_keep = [
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_type',
    'standard_value',
    'standard_units'
]
df = df[columns_to_keep]

mask = (df['standard_type'] == 'IC50') & (df['standard_value'].notna())
df = df[mask]
print("После фильтрации по IC50:", df.shape)

После фильтрации по IC50: (6979, 5)


In [ ]:
unit_map = {
    'nM': 1,
    'uM': 1000,
    'µM': 1000,
    'mM': 1_000_000
}

def convert_to_nM(value, unit):
    factor = unit_map.get(unit, None)
    if factor is not None:
        return float(value) * factor
    return None

df['standard_value'] = df.apply(lambda row: convert_to_nM(row['standard_value'], row['standard_units']), axis=1)

df = df[df['standard_value'].notna()]

print("После преобразования единиц в standard_value:", df.shape)

После преобразования единиц: (6952, 6)


In [4]:
# --- 4. Удаление дубликатов и пропусков ---

df = df.drop_duplicates(subset='canonical_smiles')
df = df.dropna(subset=['canonical_smiles'])
print("После удаления дубликатов и пустых SMILES:", df.shape)

После удаления дубликатов и пустых SMILES: (5125, 5)


In [ ]:
# --- 5. Проверка валидности SMILES ---

def is_valid_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return mol is not None

valid_mask = df['canonical_smiles'].apply(is_valid_smiles)
df = df[valid_mask]
print("После проверки валидности SMILES:", df.shape)

In [8]:
# --- Сохраняем очищенный датасет ---

df.to_csv('cox2_clean.csv', index=False)
print("Финальный датасет сохранён в 'cox2_clean.csv'")

Финальный датасет сохранён в 'cox2_clean.csv'
